# 07 — Emotion Model
### Brand Sentiment Monitor

Fine-tunes `monologg/bert-base-cased-goemotions-original` on GoEmotions
for 28-class multi-label emotion classification.

**EDA decisions implemented here:**

| Finding | Decision |
|---------|----------|
| 4  | Rare classes (<500 examples) → `pos_weight` in `BCEWithLogitsLoss` |
| 5  | Multi-label structure → sigmoid output + independent thresholds |
| 15 | Per-emotion validation — rare class F1 reported separately |
| 17 | Iterative stratified 80/10/10 split for multi-label |
| 18 | Exact hyperparams: epochs=5, batch=32, lr=3e-5, threshold=0.3 |

**Outputs:**
```
models/emotion/                      fine-tuned weights + tokenizer
models/emotion/best_checkpoint/      saved at best val macro-F1
data/kaggle/splits/emotion_test.csv  held-out test split
outputs/reports/emotion_eval.json    full evaluation report
outputs/visualizations/07_*.png      training curves + per-class F1
```

---
Run notebooks 00 → 06 first. **T4 GPU required.**

## 0. Setup

> **Before running:** Runtime → Change runtime type → **T4 GPU**

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, json, ast, warnings, random, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DRIVE_ROOT  = "/content/drive/MyDrive/brand-sentiment-monitor"
KAGGLE_PROC = os.path.join(DRIVE_ROOT, "data/kaggle/processed")
KAGGLE_SPL  = os.path.join(DRIVE_ROOT, "data/kaggle/splits")
MODEL_OUT   = os.path.join(DRIVE_ROOT, "models/emotion")
CKPT_DIR    = os.path.join(MODEL_OUT,  "best_checkpoint")
OUTPUTS_VIZ = os.path.join(DRIVE_ROOT, "outputs/visualizations")
OUTPUTS_RPT = os.path.join(DRIVE_ROOT, "outputs/reports")

for d in [KAGGLE_SPL, MODEL_OUT, CKPT_DIR, OUTPUTS_VIZ, OUTPUTS_RPT]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DRIVE_ROOT)
sys.path.insert(0, os.path.join(DRIVE_ROOT, "src"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Paths set ✅")


Mounted at /content/drive
Paths set ✅


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"
print(f"Device : {DEVICE}  ({GPU_NAME})")
if not torch.cuda.is_available():
    print("⚠️  No GPU — switch to T4 GPU runtime.")


Device : cuda  (Tesla T4)


## 1. Load Config from EDA

In [3]:
with open(os.path.join(OUTPUTS_RPT, "feature_insights.json")) as f:
    fi = json.load(f)

with open(os.path.join(OUTPUTS_RPT, "eda_metadata.json")) as f:
    meta = json.load(f)

cfg = fi["training_config"]["emotion"]

BASE_MODEL   = fi["recommended_model"]["emotion"]
EPOCHS       = int(cfg["epochs"])
BATCH_SIZE   = int(cfg["batch_size"])
LR           = float(cfg["lr"])
MAX_LENGTH   = int(cfg["max_length"])
THRESHOLD    = float(cfg["threshold"])
POS_WEIGHTS  = cfg["pos_weights"]   # dict: emotion → weight
RARE_CLASSES = meta["goemotions"]["rare_classes"]

EMOTION_LABELS = [
    "admiration","amusement","anger","annoyance","approval","caring","confusion",
    "curiosity","desire","disappointment","disapproval","disgust","embarrassment",
    "excitement","fear","gratitude","grief","joy","love","nervousness","optimism",
    "pride","realization","relief","remorse","sadness","surprise","neutral",
]
NUM_LABELS = len(EMOTION_LABELS)   # 28

print("Config loaded from feature_insights.json ✅")
print(f"  base_model   : {BASE_MODEL}")
print(f"  epochs       : {EPOCHS}")
print(f"  batch_size   : {BATCH_SIZE}")
print(f"  lr           : {LR}")
print(f"  max_length   : {MAX_LENGTH}")
print(f"  threshold    : {THRESHOLD}  (sigmoid output cutoff)")
print(f"  num_labels   : {NUM_LABELS}  (28 GoEmotions classes)")
print(f"  rare_classes : {len(RARE_CLASSES)} classes with <500 examples → get pos_weight boost")


Config loaded from feature_insights.json ✅
  base_model   : monologg/bert-base-cased-goemotions-original
  epochs       : 5
  batch_size   : 32
  lr           : 3e-05
  max_length   : 128
  threshold    : 0.3  (sigmoid output cutoff)
  num_labels   : 28  (28 GoEmotions classes)
  rare_classes : 5 classes with <500 examples → get pos_weight boost


## 2. Load Data and Build Multi-label Matrix

In [4]:
ge = pd.read_csv(os.path.join(KAGGLE_PROC, "goemotions_clean.csv"))
ge["label_ids"] = ge["label_ids"].apply(
    lambda v: ast.literal_eval(str(v)) if pd.notna(v) else []
)
ge["text"] = ge["text"].astype(str)

# Build binary label matrix: shape (N, 28)
def build_label_matrix(df, emotion_labels):
    n = len(df)
    m = len(emotion_labels)
    mat = np.zeros((n, m), dtype=np.float32)
    for i, ids in enumerate(df["label_ids"]):
        for idx in ids:
            if 0 <= idx < m:
                mat[i, idx] = 1.0
    return mat

label_matrix = build_label_matrix(ge, EMOTION_LABELS)

print(f"GoEmotions loaded : {len(ge):,} rows")
print(f"Label matrix shape: {label_matrix.shape}")
print(f"Avg labels per row: {label_matrix.sum(axis=1).mean():.2f}")
print()
print("Label distribution (top 10 by count):")
counts = label_matrix.sum(axis=0)
for i in np.argsort(counts)[::-1][:10]:
    print(f"  {EMOTION_LABELS[i]:<18} {int(counts[i]):>6,}  ({counts[i]/len(ge)*100:.1f}%)")


GoEmotions loaded : 52,294 rows
Label matrix shape: (52294, 28)
Avg labels per row: 1.18

Label distribution (top 10 by count):
  neutral            16,946  (32.4%)
  admiration          4,945  (9.5%)
  approval            3,594  (6.9%)
  gratitude           3,191  (6.1%)
  annoyance           3,036  (5.8%)
  amusement           2,792  (5.3%)
  curiosity           2,664  (5.1%)
  disapproval         2,543  (4.9%)
  love                2,461  (4.7%)
  optimism            1,957  (3.7%)


## 3. Iterative Stratified Split

Finding 17: multi-label data needs iterative stratification
(standard stratified split doesn't handle multi-label correctly).

In [5]:
# Install iterative-stratification for multi-label split
import subprocess
subprocess.run(["pip", "install", "-q", "iterative-stratification"], check=True)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

msss_outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
msss_inner = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)

idx_all = np.arange(len(ge))

# 80/20 outer split
for train_idx, temp_idx in msss_outer.split(idx_all.reshape(-1,1), label_matrix):
    pass

# 50/50 of temp → val and test
for val_idx, test_idx in msss_inner.split(temp_idx.reshape(-1,1), label_matrix[temp_idx]):
    val_idx  = temp_idx[val_idx]
    test_idx = temp_idx[test_idx]

train_ge  = ge.iloc[train_idx].reset_index(drop=True)
val_ge    = ge.iloc[val_idx].reset_index(drop=True)
test_ge   = ge.iloc[test_idx].reset_index(drop=True)
train_lbl = label_matrix[train_idx]
val_lbl   = label_matrix[val_idx]
test_lbl  = label_matrix[test_idx]

for name, df in [("train", train_ge), ("val", val_ge), ("test", test_ge)]:
    print(f"  {name:<6} {len(df):>6,} rows")

# Save test split
test_ge.to_csv(os.path.join(KAGGLE_SPL, "emotion_test.csv"), index=False)
print("\nSplits saved → data/kaggle/splits/emotion_test.csv")


  train  41,831 rows
  val     5,232 rows
  test    5,231 rows

Splits saved → data/kaggle/splits/emotion_test.csv


## 4. Dataset & DataLoader

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f"Tokenizer: {BASE_MODEL}  |  vocab: {tokenizer.vocab_size:,}")

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True, max_length=max_length,
            padding="max_length", return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.encodings["input_ids"][idx],
            "attention_mask" : self.encodings["attention_mask"][idx],
            "labels"         : self.labels[idx],
        }

print("Building datasets...")
train_ds = EmotionDataset(train_ge["text"], train_lbl, tokenizer, MAX_LENGTH)
val_ds   = EmotionDataset(val_ge["text"],   val_lbl,   tokenizer, MAX_LENGTH)
test_ds  = EmotionDataset(test_ge["text"],  test_lbl,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
print("DataLoaders ready ✅")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Tokenizer: monologg/bert-base-cased-goemotions-original  |  vocab: 28,996
Building datasets...
Train: 41,831 | Val: 5,232 | Test: 5,231
DataLoaders ready ✅


## 5. Model & Loss

`BCEWithLogitsLoss` with `pos_weight` tensor for rare class boost (Finding 4).

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

# Build pos_weight tensor from EDA-computed weights
# pos_weights dict keys are emotion label strings
pw_list = []
for emo in EMOTION_LABELS:
    pw = POS_WEIGHTS.get(emo, 1.0) if POS_WEIGHTS else 1.0
    pw_list.append(float(pw))
pos_weight_tensor = torch.tensor(pw_list, dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.06)
optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

total_p = sum(p.numel() for p in model.parameters())
print(f"Model  : {BASE_MODEL}")
print(f"Params : {total_p:,}")
print(f"Loss   : BCEWithLogitsLoss with pos_weight (rare class boost)")
print(f"Steps  : {total_steps:,}  |  warmup: {warmup_steps:,}")
print()
print("pos_weight sample (rare classes get higher weight):")
rare_set = set(RARE_CLASSES) if RARE_CLASSES else set()
for emo, pw in zip(EMOTION_LABELS, pw_list):
    if pw > 2.0 or emo in rare_set:
        print(f"  {emo:<18} pw={pw:.2f}  {'← rare class' if emo in rare_set else ''}")


pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model  : monologg/bert-base-cased-goemotions-original
Params : 108,331,804
Loss   : BCEWithLogitsLoss with pos_weight (rare class boost)
Steps  : 6,540  |  warmup: 392

pos_weight sample (rare classes get higher weight):
  admiration         pw=9.59  
  amusement          pw=17.74  
  anger              pw=26.69  
  annoyance          pw=16.54  
  approval           pw=13.72  
  caring             pw=38.46  
  confusion          pw=31.43  
  curiosity          pw=18.93  
  desire             pw=66.74  
  disappointment     pw=33.28  
  disapproval        pw=20.02  
  disgust            pw=52.57  
  embarrassment      pw=143.70  ← rare class
  excitement         pw=50.58  
  fear               pw=70.02  
  gratitude          pw=15.09  
  grief              pw=564.24  ← rare class
  joy                pw=29.40  
  love               pw=20.06  
  nervousness        pw=259.88  ← rare class
  optimism           pw=26.46  
  pride              pw=381.13  ← rare class
  realization        pw=

## 6. Training Loop

In [8]:
def run_epoch_emotion(model, loader, optimizer, scheduler, criterion, device, is_train, threshold=0.3):
    model.train() if is_train else model.eval()
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for batch in tqdm(loader, desc="train" if is_train else "eval ", leave=False):
            input_ids   = batch["input_ids"].to(device)
            attn_mask   = batch["attention_mask"].to(device)
            labels_gpu  = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            loss    = criterion(outputs.logits, labels_gpu)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            preds = (torch.sigmoid(outputs.logits) >= threshold).float()
            total_loss += loss.item() * labels_gpu.size(0)
            total_n    += labels_gpu.size(0)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels_gpu.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    avg_loss   = total_loss / total_n
    macro_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    micro_f1   = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    return avg_loss, macro_f1, micro_f1, all_preds, all_labels

print("run_epoch_emotion() defined ✅")


run_epoch_emotion() defined ✅


In [9]:
history = {"train_loss":[], "val_loss":[], "train_macro_f1":[], "val_macro_f1":[], "val_micro_f1":[]}
best_val_f1 = 0.0
best_epoch  = 0

print(f"Training — {EPOCHS} epoch(s)")
print(f"{'Ep':<4} {'TrLoss':>8} {'VaLoss':>8} {'TrMacF1':>9} {'VaMacF1':>9} {'VaMicF1':>9}  Note")
print("─" * 65)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_mac, tr_mic, _, _ = run_epoch_emotion(
        model, train_loader, optimizer, scheduler, criterion, DEVICE, True, THRESHOLD
    )
    va_loss, va_mac, va_mic, val_preds, val_labels = run_epoch_emotion(
        model, val_loader, optimizer, scheduler, criterion, DEVICE, False, THRESHOLD
    )

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_macro_f1"].append(tr_mac)
    history["val_macro_f1"].append(va_mac)
    history["val_micro_f1"].append(va_mic)

    is_best = va_mac > best_val_f1
    if is_best:
        best_val_f1 = va_mac
        best_epoch  = epoch
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)

    note = "✅ best saved" if is_best else ""
    print(f"  {epoch:<2}  {tr_loss:>8.4f} {va_loss:>8.4f} {tr_mac:>9.4f} {va_mac:>9.4f} {va_mic:>9.4f}  {note}")

print(f"\nBest val macro F1 = {best_val_f1:.4f} at epoch {best_epoch}")


Training — 5 epoch(s)
Ep     TrLoss   VaLoss   TrMacF1   VaMacF1   VaMicF1  Note
─────────────────────────────────────────────────────────────────


train:   0%|          | 0/1308 [00:00<?, ?it/s]

eval :   0%|          | 0/164 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  1     0.3750   0.2814    0.4971    0.5073    0.6099  ✅ best saved


...

  4     0.0829   0.4276    0.7415    0.6726    0.7633  ✅ best saved


train:   0%|          | 0/1308 [00:00<?, ?it/s]

eval :   0%|          | 0/164 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  5     0.0590   0.4605    0.7982    0.7078    0.7919  ✅ best saved

Best val macro F1 = 0.7078 at epoch 5


## 7. Training Curves

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_x = list(range(1, EPOCHS + 1))

axes[0].plot(ep_x, history["train_loss"],     "o-", label="Train", color="#3498db")
axes[0].plot(ep_x, history["val_loss"],       "o-", label="Val",   color="#e74c3c")
axes[0].set_title("BCE Loss per Epoch", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCEWithLogitsLoss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_x, history["train_macro_f1"], "o-", label="Train Macro F1", color="#3498db")
axes[1].plot(ep_x, history["val_macro_f1"],   "o-", label="Val Macro F1",   color="#e74c3c")
axes[1].plot(ep_x, history["val_micro_f1"],   "s--",label="Val Micro F1",   color="#9b59b6", alpha=0.8)
axes[1].axhline(0.70, color="gray", ls=":", lw=1.5, label="Target ≥ 0.70")
axes[1].set_title("F1 per Epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Emotion Model — Training History", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "07_training_curves.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/07_training_curves.png


## 8. Test Set Evaluation + Per-Class F1

In [11]:
best_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(DEVICE)
best_model.eval()

all_preds_t, all_labels_t = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="test set"):
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        labels_cpu = batch["labels"]

        outputs = best_model(input_ids=input_ids, attention_mask=attn_mask)
        preds   = (torch.sigmoid(outputs.logits) >= THRESHOLD).float()

        all_preds_t.append(preds.cpu().numpy())
        all_labels_t.append(labels_cpu.numpy())

all_preds_t  = np.vstack(all_preds_t)
all_labels_t = np.vstack(all_labels_t)

macro_f1 = f1_score(all_labels_t, all_preds_t, average="macro",  zero_division=0)
micro_f1 = f1_score(all_labels_t, all_preds_t, average="micro",  zero_division=0)
samp_f1  = f1_score(all_labels_t, all_preds_t, average="samples",zero_division=0)

print(f"Test Set Results:")
print(f"  Macro F1  : {macro_f1:.4f}  {'✅ ≥ 0.70' if macro_f1 >= 0.70 else '⚠️  < 0.70'}")
print(f"  Micro F1  : {micro_f1:.4f}")
print(f"  Sample F1 : {samp_f1:.4f}")

# Per-class F1
per_class_f1 = f1_score(all_labels_t, all_preds_t, average=None, zero_division=0)
print()
print(f"Per-class F1 (sorted worst→best):")
emo_f1_sorted = sorted(zip(EMOTION_LABELS, per_class_f1), key=lambda x: x[1])
for emo, f1 in emo_f1_sorted:
    rare_flag = " ← rare" if emo in (RARE_CLASSES or []) else ""
    bar = "█" * int(f1 * 20)
    print(f"  {emo:<18} {f1:.4f}  {bar}{rare_flag}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

test set:   0%|          | 0/164 [00:00<?, ?it/s]

Test Set Results:
  Macro F1  : 0.7021  ✅ ≥ 0.70
  Micro F1  : 0.7834
  Sample F1 : 0.8374

Per-class F1 (sorted worst→best):
  grief              0.2500  █████ ← rare
  relief             0.4918  █████████ ← rare
  nervousness        0.5263  ██████████ ← rare
  pride              0.5854  ███████████ ← rare
  realization        0.6011  ████████████
  excitement         0.6345  ████████████
  disappointment     0.6364  ████████████
  disgust            0.6691  █████████████
  embarrassment      0.6863  █████████████ ← rare
  desire             0.6887  █████████████
  annoyance          0.6906  █████████████
  caring             0.7035  ██████████████
  disapproval        0.7042  ██████████████
  sadness            0.7064  ██████████████
  confusion          0.7079  ██████████████
  joy                0.7133  ██████████████
  anger              0.7164  ██████████████
  approval           0.7329  ██████████████
  fear               0.7368  ██████████████
  surprise           0.7382  █████

In [12]:
# Per-class F1 bar chart
fig, ax = plt.subplots(figsize=(14, 8))

emos_sorted = [e for e, _ in emo_f1_sorted]
f1s_sorted  = [f for _, f in emo_f1_sorted]
colors_bar  = ["#e74c3c" if e in (RARE_CLASSES or []) else "#3498db" for e in emos_sorted]

bars = ax.barh(emos_sorted, f1s_sorted, color=colors_bar, alpha=0.85, edgecolor="white")
ax.axvline(0.70, color="black", ls="--", lw=1.5, label="Target 0.70")
ax.axvline(macro_f1, color="#e74c3c", ls=":", lw=2, label=f"Macro avg {macro_f1:.3f}")
ax.set_xlabel("F1 Score"); ax.set_title("Emotion Model — Per-Class F1 (Test Set)", fontweight="bold")
ax.legend()

# Annotate bars
for bar, f1 in zip(bars, f1s_sorted):
    ax.text(f1 + 0.005, bar.get_y() + bar.get_height()/2,
            f"{f1:.3f}", va="center", fontsize=8)

# Legend for rare classes
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#3498db", label="Normal class"),
    Patch(color="#e74c3c", label="Rare class (<500 examples)"),
    plt.Line2D([0],[0], color="black", ls="--", label="Target 0.70"),
    plt.Line2D([0],[0], color="#e74c3c", ls=":", label=f"Macro F1={macro_f1:.3f}"),
])

plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "07_per_class_f1.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/07_per_class_f1.png


## 9. Save Final Model

In [13]:
for fname in os.listdir(MODEL_OUT):
    fpath = os.path.join(MODEL_OUT, fname)
    if os.path.isfile(fpath):
        os.remove(fpath)

for fname in os.listdir(CKPT_DIR):
    shutil.copy2(os.path.join(CKPT_DIR, fname), os.path.join(MODEL_OUT, fname))

saved = sorted(os.listdir(MODEL_OUT))
print(f"models/emotion/ — {len(saved)} files:")
for fname in saved:
    sz = os.path.getsize(os.path.join(MODEL_OUT, fname)) / 1e6
    print(f"  {fname:<45}  {sz:>7.1f} MB")
print("✅ Model saved")


models/emotion/ — 5 files:
  best_checkpoint                                    0.0 MB
  config.json                                        0.0 MB
  model.safetensors                                433.4 MB
  tokenizer.json                                     0.7 MB
  tokenizer_config.json                              0.0 MB
✅ Model saved


In [14]:
from models.emotion import EmotionModel

emotion_model = EmotionModel.load(MODEL_OUT)

verify_cases = [
    ("I am absolutely furious with this terrible product",         "anger"),
    ("Just got my new shoes and I am so happy with them",          "joy"),
    ("Feeling overwhelmed and scared about the situation",         "fear"),
    ("This is the saddest thing I have ever experienced",          "sadness"),
    ("I am so grateful for the amazing customer service",          "gratitude"),
]

print("End-to-end verification (EmotionModel.load → predict):")
print(f"  {'Expected':>12} {'Top Emotion':>12} {'Score':>7}  {'OK':>4}  Text")
print("  " + "─" * 75)
all_ok = True
for text, expected_top in verify_cases:
    result  = emotion_model.predict(text)
    correct = result["top_emotion"] == expected_top
    all_ok  = all_ok and correct
    status  = "✅" if correct else "❌ (acceptable — emotions overlap)"
    print(f"  {expected_top:>12} {result['top_emotion']:>12} {result['all_scores'].get(expected_top, 0):>7.4f}  {status}")
    print(f"    Active: {list(result['emotions'].keys())}")

print()
print("✅ Emotion model working end-to-end" if all_ok else "ℹ️  Some top emotions differ — emotions can overlap and model may be correct")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

End-to-end verification (EmotionModel.load → predict):
      Expected  Top Emotion   Score    OK  Text
  ───────────────────────────────────────────────────────────────────────────
         anger        anger  0.9984  ✅
    Active: ['anger']
           joy          joy  0.9995  ✅
    Active: ['joy']
          fear         fear  0.9995  ✅
    Active: ['fear']
       sadness      sadness  0.9994  ✅
    Active: ['sadness']
     gratitude    gratitude  0.9972  ✅
    Active: ['admiration', 'gratitude']

✅ Emotion model working end-to-end


## 10. Save Evaluation Report

In [15]:
per_class_report = {
    emo: {
        "f1"      : round(float(f1), 4),
        "support" : int(all_labels_t[:, i].sum()),
        "is_rare" : emo in (RARE_CLASSES or []),
    }
    for i, (emo, f1) in enumerate(zip(EMOTION_LABELS, per_class_f1))
}

rare_f1s   = [v["f1"] for v in per_class_report.values() if v["is_rare"]]
common_f1s = [v["f1"] for v in per_class_report.values() if not v["is_rare"]]

emotion_report = {
    "model_path"  : "models/emotion/",
    "base_model"  : BASE_MODEL,
    "num_labels"  : NUM_LABELS,
    "emotion_labels": EMOTION_LABELS,
    "training": {
        "epochs"    : EPOCHS,
        "best_epoch": best_epoch,
        "batch_size": BATCH_SIZE,
        "lr"        : LR,
        "max_length": MAX_LENGTH,
        "threshold" : THRESHOLD,
        "n_train"   : len(train_ge),
        "n_val"     : len(val_ge),
        "n_test"    : len(test_ge),
    },
    "val_best"    : {"macro_f1": round(best_val_f1, 4), "epoch": best_epoch},
    "test_results": {
        "macro_f1"  : round(macro_f1, 4),
        "micro_f1"  : round(micro_f1, 4),
        "sample_f1" : round(samp_f1, 4),
        "per_class" : per_class_report,
        "rare_class_avg_f1"  : round(float(np.mean(rare_f1s)), 4) if rare_f1s else None,
        "common_class_avg_f1": round(float(np.mean(common_f1s)), 4) if common_f1s else None,
    },
    "target_met"      : macro_f1 >= 0.70,
    "training_history": history,
    "outputs": {
        "model_dir"          : "models/emotion/",
        "training_curves_png": "outputs/visualizations/07_training_curves.png",
        "per_class_f1_png"   : "outputs/visualizations/07_per_class_f1.png",
        "eval_json"          : "outputs/reports/emotion_eval.json",
        "test_split_csv"     : "data/kaggle/splits/emotion_test.csv",
    },
}

rpt_path = os.path.join(OUTPUTS_RPT, "emotion_eval.json")
with open(rpt_path, "w") as f:
    json.dump(emotion_report, f, indent=2)

print(f"Saved → {rpt_path}")
print()
print("=" * 60)
print("EMOTION MODEL — FINAL SUMMARY")
print("=" * 60)
print(f"  Base model       : {BASE_MODEL}")
print(f"  Best epoch       : {best_epoch} / {EPOCHS}")
print(f"  Val macro F1     : {best_val_f1:.4f}")
print(f"  Test macro F1    : {macro_f1:.4f}  {'✅ ≥ 0.70' if macro_f1 >= 0.70 else '⚠️  < 0.70'}")
print(f"  Test micro F1    : {micro_f1:.4f}")
print(f"  Rare class avg F1: {np.mean(rare_f1s):.4f}" if rare_f1s else "  No rare class data")
print(f"  Model saved      : models/emotion/")
print("=" * 60)
print(f"\n✅ Notebook 07 complete. Next: 08_topic_model.ipynb")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/reports/emotion_eval.json

EMOTION MODEL — FINAL SUMMARY
  Base model       : monologg/bert-base-cased-goemotions-original
  Best epoch       : 5 / 5
  Val macro F1     : 0.7078
  Test macro F1    : 0.7021  ✅ ≥ 0.70
  Test micro F1    : 0.7834
  Rare class avg F1: 0.5080
  Model saved      : models/emotion/

✅ Notebook 07 complete. Next: 08_topic_model.ipynb
